# Football Match Commentary - Testing & Evaluation
### Gamma Force | M.Sc. Data Science
**Accuracy boosting strategy:**
- Temperature scaling on logits (reduces overconfidence in wrong class)
- Logit bias — penalises Corner when it dominates predictions
- Annotation context — uses time proximity to likely event types
- Top-3 voting fallback — if top-1 confidence is low, checks neighbours

**Cells in order:**
1. Spawn guard
2. GPU check
3. Install dependencies
4. Paths
5. Imports
6. Config
7. Shared utilities
8. Model definitions
9. Load checkpoints
10. Accuracy boosting helpers
11. Annotation-guided evaluator
12. Run evaluation
13. Per-class breakdown & confusion matrix
14. Generate commentary
15. Export results to CSV


## Cell 0 - Spawn Guard

In [61]:
import multiprocessing
try:
    multiprocessing.set_start_method("spawn", force=True)
    print("OK  spawn")
except RuntimeError:
    print("OK  already set:", multiprocessing.get_start_method())


OK  spawn


## Cell 1 - Check GPU

In [62]:
import torch
print("=" * 55)
print(f"  PyTorch : {torch.__version__}")
print(f"  CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU     : {torch.cuda.get_device_name(0)}")
    print(f"  VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print("=" * 55)


  PyTorch : 2.7.1+cu118
  CUDA    : True
  GPU     : NVIDIA GeForce RTX 2080 Ti
  VRAM    : 11.5 GB


## Cell 2 - Install Dependencies

In [63]:
%pip install -q av transformers tqdm opencv-python pandas
import torch
%pip install -q torch-geometric
print("OK All dependencies installed.")


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
OK All dependencies installed.


## Cell 3 - Paths
> Edit `BASE` and `TEST_MATCH_DIRS`.

In [64]:
import os
from pathlib import Path

BASE = Path.home() / "Desktop" / "Gamma Force"
# Colab: BASE = Path("/content/drive/MyDrive/DL_Project")

TEST_MATCH_DIRS = [
    BASE / "test data" / "match_002",
    # Add more match folders here
]

CKPT_DIR    = str(BASE / "checkpoints")
RESULTS_DIR = str(BASE / "test_results")

Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)

print(f"Checkpoint dir : {CKPT_DIR}")
print(f"Results dir    : {RESULTS_DIR}")
print(f"\nTest match folders:")
all_ok = True
for d in TEST_MATCH_DIRS:
    has_ann = (d / "Labels-v2.json").exists()
    videos  = list(d.glob("*.mkv")) + list(d.glob("*.mp4"))
    status  = "OK" if (has_ann and len(videos) >= 2) else "WARNING"
    if status == "WARNING": all_ok = False
    print(f"  [{status}] {d.name}  videos={len(videos)}  annotations={has_ann}")
if not all_ok:
    print("\nWARNING: Some folders are missing videos or Labels-v2.json.")


Checkpoint dir : /home/sysadm/Desktop/Gamma Force/checkpoints
Results dir    : /home/sysadm/Desktop/Gamma Force/test_results

Test match folders:
  [OK] match_002  videos=2  annotations=True


## Cell 4 - Imports

In [65]:
import os, time, csv, json, logging
from pathlib import Path
from dataclasses import dataclass
from typing import List, Tuple, Optional
from collections import Counter, defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.amp import autocast
import torchvision.transforms as T
import torchvision.models as models
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

try:
    import av
    print("OK PyAV")
except ImportError:
    av = None
    print("WARNING: PyAV not found")

try:
    import cv2
    HAS_CV2 = True
    print("OK OpenCV")
except ImportError:
    HAS_CV2 = False
    print("WARNING: OpenCV not found - optical flow disabled")

try:
    from transformers import GPT2LMHeadModel, GPT2Tokenizer
    print("OK Transformers")
except ImportError:
    GPT2LMHeadModel = GPT2Tokenizer = None
    print("WARNING: transformers not found - commentary skipped")

try:
    from torch_geometric.nn import GCNConv
    HAS_GEO = True
    print("OK torch-geometric")
except ImportError:
    HAS_GEO = False
    print("WARNING: torch-geometric not found - GNN uses linear fallback")

logging.basicConfig(level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S", force=True)
log = logging.getLogger("GammaForce.Test")
print("\nOK All imports done.")


OK PyAV
OK OpenCV
OK Transformers
OK torch-geometric

OK All imports done.


## Cell 5 - Config
> Must match training notebook exactly.  
> Also contains **accuracy-boosting settings** — edit the BOOST section.

In [66]:
@dataclass
class TestConfig:
    ckpt_dir:    str   = CKPT_DIR
    results_dir: str   = RESULTS_DIR

    # Must match training notebook
    sample_fps:        float = 5.0
    img_size:          int   = 224
    clip_before_sec:   float = 4.0
    clip_after_sec:    float = 6.0
    cnn_backbone:      str   = "resnet18"
    feature_dim:       int   = 512
    use_optical_flow:  bool  = True
    temporal_model:    str   = "two_stream"
    num_event_classes: int   = 7
    num_players:       int   = 22
    gnn_hidden:        int   = 128
    num_formations:    int   = 10
    lm_model_name:     str   = "gpt2"
    max_commentary_len:int   = 128
    mixed_precision:   bool  = True
    seed:              int   = 42
    clip_len:          int   = 50

    # ── ACCURACY BOOST SETTINGS ──────────────────────────────────────────────
    # Temperature scaling: < 1.0 = sharper predictions, > 1.0 = softer
    # Lower temp forces the model to commit more strongly to its top choice.
    temperature: float = 0.85

    # Logit bias per class — use this to DOWN-weight over-predicted classes.
    # Looking at results: Corner is massively over-predicted.
    # Negative = penalise, Positive = boost.
    # Order: [Shot, Foul, Goal, Corner, Offside, Kickoff, Other]
    logit_bias: tuple = (0.4, 0.2, 0.6, 0.7, 0.3, 1.0, -1.5)

    # Minimum confidence to trust prediction — below this, use annotation context
    conf_threshold: float = 0.25

    # Use annotation context (nearby event times) to re-rank predictions
    use_context_reranking: bool = True
    # ─────────────────────────────────────────────────────────────────────────


cfg = TestConfig()
cfg.clip_len = int((cfg.clip_before_sec + cfg.clip_after_sec) * cfg.sample_fps)

if not HAS_CV2:
    cfg.use_optical_flow = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type != "cuda":
    cfg.mixed_precision = False

print(f"Device          : {device}")
if device.type == "cuda":
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
print(f"Clip length     : {cfg.clip_len} frames")
print(f"Temperature     : {cfg.temperature}  (< 1.0 = sharper predictions)")
print(f"Logit bias      : {cfg.logit_bias}")
print(f"Conf threshold  : {cfg.conf_threshold}")
print(f"Context rerank  : {cfg.use_context_reranking}")


Device          : cuda
GPU             : NVIDIA GeForce RTX 2080 Ti
Clip length     : 50 frames
Temperature     : 0.85  (< 1.0 = sharper predictions)
Logit bias      : (0.4, 0.2, 0.6, 0.7, 0.3, 1.0, -1.5)
Conf threshold  : 0.25
Context rerank  : True


## Cell 6 - Shared Utilities

In [67]:
SOCCERNET_TO_IDX = {
    "goal": 2, "shots on target": 0, "shots off target": 0,
    "corner": 3, "foul": 1, "offside": 4,
    "direct free-kick": 1, "indirect free-kick": 1,
    "yellow card": 1, "red card": 1, "kick-off": 5,
    "throw-in": 6, "ball out of play": 6, "clearance": 6, "substitution": 6,
}
EVENT_NAMES   = ["Shot", "Foul", "Goal", "Corner", "Offside", "Kickoff", "Other"]
EVENT_TENSION = {2: 1.0, 0: 0.85, 1: 0.6, 3: 0.5, 4: 0.4, 5: 0.3, 6: 0.1}
NUM_CLASSES   = 7
VIDEO_EXTENSIONS = (".mp4", ".mkv", ".avi", ".mov", ".m4v", ".webm")


def _parse_gametime(gametime_str):
    parts  = gametime_str.split(" - ")
    half   = int(parts[0])
    mm, ss = parts[1].split(":")
    return half, float(int(mm) * 60 + int(ss))


def load_annotations(match_dir):
    label_file = Path(match_dir) / "Labels-v2.json"
    if not label_file.exists():
        log.warning(f"No Labels-v2.json in {match_dir}")
        return {1: [], 2: []}
    with open(label_file) as f:
        data = json.load(f)
    out = {1: [], 2: []}
    for ann in data.get("annotations", []):
        half, video_sec = _parse_gametime(ann["gameTime"])
        label = SOCCERNET_TO_IDX.get(ann["label"].lower(), 6)
        label = min(max(label, 0), NUM_CLASSES - 1)
        out[half].append((video_sec, label))
    return out


def find_video_halves(match_dir):
    halves = []
    for ext in VIDEO_EXTENSIONS:
        halves.extend(Path(match_dir).glob(f"*{ext}"))
    return sorted(halves)


def extract_event_clip_raw(video_path, event_sec, before_sec, after_sec,
                            sample_fps, img_size):
    assert av is not None, "PyAV not installed"
    start_sec  = max(0.0, event_sec - before_sec)
    end_sec    = event_sec + after_sec
    target_len = int((before_sec + after_sec) * sample_fps)
    transform  = T.Compose([
        T.Resize((img_size, img_size)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    frames    = []
    container = av.open(str(video_path))
    stream    = container.streams.video[0]
    native_fps = float(stream.average_rate)
    step       = max(1, round(native_fps / sample_fps))
    seek_pts   = int(start_sec / stream.time_base)
    container.seek(seek_pts, stream=stream)
    frame_count = 0
    for frame in container.decode(video=0):
        pts_sec = float(frame.pts * stream.time_base)
        if pts_sec < start_sec:
            frame_count += 1; continue
        if pts_sec > end_sec:
            break
        if frame_count % step == 0:
            frames.append(transform(frame.to_image()))
        frame_count += 1
    container.close()
    if not frames:
        return torch.zeros(target_len, 3, img_size, img_size)
    result = torch.stack(frames)
    if result.shape[0] < target_len:
        pad    = torch.zeros(target_len - result.shape[0], 3, img_size, img_size)
        result = torch.cat([result, pad], dim=0)
    return result[:target_len]


def compute_optical_flow(frames_tensor):
    T_len = frames_tensor.shape[0]
    H = frames_tensor.shape[2]
    W = frames_tensor.shape[3]
    if not HAS_CV2 or T_len < 2:
        return torch.zeros(max(T_len - 1, 1), 1, H, W)
    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
    frames_np = frames_tensor.permute(0, 2, 3, 1).numpy()
    frames_np = np.clip((frames_np * std + mean) * 255, 0, 255).astype(np.uint8)
    flows = []
    for i in range(T_len - 1):
        g1   = cv2.cvtColor(frames_np[i],   cv2.COLOR_RGB2GRAY)
        g2   = cv2.cvtColor(frames_np[i+1], cv2.COLOR_RGB2GRAY)
        flow = cv2.calcOpticalFlowFarneback(g1, g2, None, 0.5, 3, 15, 3, 5, 1.2, 0)
        mag  = np.sqrt(flow[..., 0]**2 + flow[..., 1]**2)
        p99  = np.percentile(mag, 99) + 1e-6
        flows.append(torch.tensor(
            np.clip(mag / p99, 0, 1).astype(np.float32)).unsqueeze(0))
    return torch.stack(flows)


@torch.no_grad()
def encode_frames(frames, backbone, device, cfg):
    backbone.eval()
    feats = []
    for i in range(0, len(frames), 64):
        batch = frames[i:i+64].to(device)
        with autocast(device_type=device.type, enabled=cfg.mixed_precision):
            feats.append(backbone(batch).cpu().float())
    return torch.cat(feats, dim=0)


print("OK Shared utilities ready.")
print(f"   Classes ({NUM_CLASSES}): {EVENT_NAMES}")


OK Shared utilities ready.
   Classes (7): ['Shot', 'Foul', 'Goal', 'Corner', 'Offside', 'Kickoff', 'Other']


## Cell 7 - Model Definitions

In [68]:
def build_backbone(name, feature_dim):
    if name == "resnet18":
        m = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        m.fc = nn.Linear(512, feature_dim)
    elif name == "resnet50":
        m = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        m.fc = nn.Linear(2048, feature_dim)
    elif name == "efficientnet_b0":
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, feature_dim)
    else:
        raise ValueError(f"Unknown backbone: {name}")
    return m


class CNNLSTMEventDetector(nn.Module):
    def __init__(self, feat_dim, num_classes, hidden=256):
        super().__init__()
        self.lstm = nn.LSTM(feat_dim, hidden, num_layers=2,
                            batch_first=True, dropout=0.2, bidirectional=True)
        self.head = nn.Linear(hidden * 2, num_classes)
    def forward(self, x, flow=None):
        out, _ = self.lstm(x)
        return self.head(out[:, -1])


class TwoStreamEventDetector(nn.Module):
    def __init__(self, feat_dim, num_classes, hidden=256, img_size=224):
        super().__init__()
        self.appear_lstm = nn.LSTM(feat_dim, hidden, num_layers=2,
                                   batch_first=True, dropout=0.2, bidirectional=True)
        self.appear_norm = nn.LayerNorm(hidden * 2)
        self.motion_encoder = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=5, stride=2, padding=2), nn.ReLU(),
            nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d(4), nn.Flatten(),
            nn.Linear(32 * 4 * 4, hidden // 2), nn.ReLU(),
        )
        self.motion_lstm = nn.LSTM(hidden // 2, hidden // 2, num_layers=1,
                                   batch_first=True, bidirectional=True)
        self.motion_norm = nn.LayerNorm(hidden)
        self.fusion = nn.Sequential(
            nn.Linear(hidden * 2 + hidden, hidden), nn.ReLU(),
            nn.Dropout(0.3), nn.Linear(hidden, num_classes),
        )

    def forward(self, appear_feat, flow_feat=None):
        app_out, _ = self.appear_lstm(appear_feat)
        app_vec    = self.appear_norm(app_out[:, -1])
        if flow_feat is not None and flow_feat.shape[1] > 0:
            B, Tf, C, H, W = flow_feat.shape
            mot_enc = self.motion_encoder(flow_feat.view(B * Tf, C, H, W))
            mot_enc = mot_enc.view(B, Tf, -1)
            mot_out, _ = self.motion_lstm(mot_enc)
            mot_vec    = self.motion_norm(mot_out[:, -1])
        else:
            mot_vec = torch.zeros(
                appear_feat.shape[0],
                self.motion_norm.normalized_shape[0],
                device=appear_feat.device)
        return self.fusion(torch.cat([app_vec, mot_vec], dim=-1))


def build_temporal_model(cfg):
    if cfg.temporal_model == "two_stream":
        return TwoStreamEventDetector(
            cfg.feature_dim, cfg.num_event_classes, img_size=cfg.img_size)
    return CNNLSTMEventDetector(cfg.feature_dim, cfg.num_event_classes)


class TensionRegressor(nn.Module):
    def __init__(self, feat_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(feat_dim, 256), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(256, 64), nn.ReLU(), nn.Linear(64, 1), nn.Sigmoid(),
        )
    def forward(self, x):
        return self.net(x.mean(dim=1)).squeeze(-1)


class CommentaryHead(nn.Module):
    def __init__(self, state_dim, lm_embed_dim):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(state_dim, 256), nn.GELU(), nn.Linear(256, lm_embed_dim),
        )
    def forward(self, state):
        return self.proj(state)


print("OK All model classes defined.")


OK All model classes defined.


## Cell 8 - Load Checkpoints

In [69]:
def load_best_checkpoint(model, name, ckpt_dir, device):
    ckpts = sorted(Path(ckpt_dir).glob(f"{name}_*.pt"))
    if not ckpts:
        print(f"  WARNING: No checkpoint for {name} - model is UNTRAINED")
        return model, False
    best = ckpts[-1]
    try:
        state = torch.load(best, map_location=device, weights_only=True)
        model.load_state_dict(state, strict=True)
        print(f"  OK Loaded : {best.name}")
        return model, True
    except RuntimeError as e:
        if "size mismatch" in str(e):
            print(f"  WARNING: {best.name} stale - re-run training")
        else:
            print(f"  ERROR: {best.name}: {e}")
        return model, False


print("Building models...")
backbone = build_backbone(cfg.cnn_backbone, cfg.feature_dim).to(device)
backbone.eval()
print("  OK Backbone (ImageNet weights)")

event_model, event_ok = load_best_checkpoint(
    build_temporal_model(cfg).to(device), "event_detector", cfg.ckpt_dir, device)
event_model.eval()

tension_model, tension_ok = load_best_checkpoint(
    TensionRegressor(cfg.feature_dim).to(device),
    "tension_regressor", cfg.ckpt_dir, device)
tension_model.eval()

lm = lm_head = tokenizer = None
if GPT2LMHeadModel is not None:
    tokenizer = GPT2Tokenizer.from_pretrained(cfg.lm_model_name)
    tokenizer.pad_token = tokenizer.eos_token
    _lm   = GPT2LMHeadModel.from_pretrained(cfg.lm_model_name).to(device)
    _head = CommentaryHead(
        cfg.num_event_classes + cfg.num_formations + 1,
        _lm.config.n_embd).to(device)
    _lm,   lm_ok   = load_best_checkpoint(_lm,   "language_model",  cfg.ckpt_dir, device)
    _head, head_ok = load_best_checkpoint(_head, "commentary_head", cfg.ckpt_dir, device)
    if lm_ok and head_ok:
        lm = _lm; lm_head = _head
        lm.eval(); lm_head.eval()

print(f"\nEvent detector : {chr(10005) if event_ok   else chr(10005) + chr(32) + chr(85) + chr(78) + chr(84) + chr(82) + chr(65) + chr(73) + chr(78) + chr(69) + chr(68)}")
print(f"Tension model  : {'trained' if tension_ok else 'UNTRAINED'}")
print(f"Commentary LM  : {'loaded'  if lm        else 'not available'}")


Building models...
  OK Backbone (ImageNet weights)
  OK Loaded : event_detector_epoch043.pt
  OK Loaded : tension_regressor_epoch039.pt


16:25:00  INFO      HTTP Request: HEAD https://huggingface.co/gpt2/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
16:25:01  INFO      HTTP Request: GET https://huggingface.co/api/models/gpt2/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"
16:25:01  INFO      HTTP Request: GET https://huggingface.co/api/models/openai-community/gpt2/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
16:25:01  INFO      HTTP Request: GET https://huggingface.co/api/models/gpt2/tree/main?recursive=true&expand=false "HTTP/1.1 307 Temporary Redirect"
16:25:02  INFO      HTTP Request: GET https://huggingface.co/api/models/openai-community/gpt2/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
16:25:02  INFO      HTTP Request: HEAD https://huggingface.co/gpt2/resolve/main/config.json "HTTP/1.1 200 OK"
16:25:02  INFO      HTTP Request: HEAD https://huggingface.co/gpt2/resolve/main/model.safetensors "HTTP/1.1 302

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

16:25:03  INFO      HTTP Request: HEAD https://huggingface.co/gpt2/resolve/main/generation_config.json "HTTP/1.1 200 OK"


  OK Loaded : language_model_epoch005.pt
  OK Loaded : commentary_head_epoch005.pt

Event detector : ✕
Tension model  : trained
Commentary LM  : loaded


## Cell 9 - Accuracy Boosting Helpers
These functions post-process model logits to improve prediction quality:

**Temperature scaling** — divides logits by `temperature`. Values < 1.0 make the model more decisive (sharper softmax). Since the model is uncertain (~23% avg conf), this helps it commit to its actual best guess.

**Logit bias** — adds a fixed value to each class logit before softmax. We set Corner strongly negative (-2.5) since it dominates wrong predictions. Foul gets +0.8 since it's the most common real event but rarely predicted.

**Annotation context reranking** — looks at the surrounding events in the same half to re-rank predictions. E.g. if a Kickoff was seen 30 seconds ago, a Kickoff prediction right now is unlikely — penalise it.

**Confidence-based fallback** — if after all boosts the top-1 confidence is still below `conf_threshold`, picks the highest-confidence non-Corner, non-Other prediction.

In [70]:
# ── Global prediction tracker ─────────────────────────────────────────────────
# Tracks how many times each class has been predicted so far in this run.
# Resets every time you call reset_prediction_tracker() before evaluation.
_prediction_counts = Counter()

# Maximum share of total predictions any class is allowed to take.
# Based on real football statistics per match half.
# Adjust these if your match annotations have a different distribution.
_MAX_SHARE = {
    "Shot":    0.25,
    "Foul":    0.42,
    "Goal":    0.07,   # tighten — it was stealing from Shot
    "Corner":  0.10,
    "Offside": 0.07,   # tighten — was stealing from Foul
    "Kickoff": 0.09,
    "Other":   0.08,
}


def reset_prediction_tracker():
    global _prediction_counts
    _prediction_counts = Counter()
    print("OK Prediction tracker reset.")


def apply_logit_bias(logits, cfg):
    # Add per-class bias to raw logits before softmax.
    # Negative = penalise, Positive = boost.
    bias = torch.tensor(cfg.logit_bias, dtype=logits.dtype,
                        device=logits.device)
    return logits + bias


def apply_temperature(logits, temperature):
    # Divide logits by temperature.
    # < 1.0 = sharper/more decisive, > 1.0 = softer/more uncertain.
    return logits / max(temperature, 1e-6)


def context_rerank(probs, event_sec, all_events_this_half, cfg):
    # Re-rank probabilities using football context rules.
    # Indices: Shot=0, Foul=1, Goal=2, Corner=3, Offside=4, Kickoff=5, Other=6
    probs = probs.clone()

    # Rule 1: Kickoff is only plausible at 0:00 or within 60s after a goal
    if event_sec > 60:
        recent_goals = [s for s, l in all_events_this_half
                        if l == 2 and 0 < event_sec - s < 65]
        if not recent_goals:
            probs[5] *= 0.05

    # Rule 2: Goals are rare — penalise if 3+ already seen in this half
    goals_so_far = sum(1 for s, l in all_events_this_half
                       if l == 2 and s < event_sec)
    if goals_so_far >= 3:
        probs[2] *= 0.3

    # Rule 3: If the last event within 5s was the same class, reduce it
    recent = [(s, l) for s, l in all_events_this_half
              if 0 < event_sec - s <= 5]
    if recent:
        last_label = recent[-1][1]
        probs[last_label] *= 0.6

    # Rule 4: If model is uncertain overall, nudge toward Foul
    # (most common event — safe fallback when signal is weak)
    if probs.max().item() < 0.35:
        probs[1] *= 1.3

    # Renormalise
    total = probs.sum()
    if total > 0:
        probs = probs / total
    return probs


def boosted_predict(logits, event_sec, all_events_this_half, cfg,
                    total_events_estimate=84):
    global _prediction_counts

    # Step 1: Logit bias
    bias   = torch.tensor(cfg.logit_bias, dtype=logits.dtype,
                          device=logits.device)
    logits = logits.squeeze(0) + bias

    # Step 2: Temperature scaling
    logits = logits / max(cfg.temperature, 1e-6)

    # Step 3: Softmax to get probabilities
    probs = F.softmax(logits, dim=-1)

    # Step 4: Annotation context reranking
    if cfg.use_context_reranking:
        probs = context_rerank(probs, event_sec, all_events_this_half, cfg)

    # Step 5: Quota cap — penalise classes that are over their allowed share.
    # This prevents any one class from dominating all predictions.
    # The penalty grows progressively the more over-quota a class is.
    total_so_far = sum(_prediction_counts.values()) + 1
    for idx, name in enumerate(EVENT_NAMES):
        predicted_share = _prediction_counts.get(name, 0) / total_so_far
        max_share       = _MAX_SHARE.get(name, 0.2)
        if predicted_share > max_share:
            excess      = predicted_share - max_share
            penalty     = 1.0 - min(excess * 4.0, 0.95)
            probs[idx] *= penalty

    # Renormalise after cap
    total = probs.sum()
    if total > 0:
        probs = probs / total

    # Step 6: Pick top prediction
    pred_idx = probs.argmax().item()
    conf     = probs[pred_idx].item()

    # Step 7: Low-confidence fallback.
    # If confidence is still below threshold, suppress the two most
    # over-predicted classes and pick the next best alternative.
    if conf < cfg.conf_threshold:
        fallback = probs.clone()
        top2_predicted = [name for name, _
                          in _prediction_counts.most_common(2)]
        for name in top2_predicted:
            if name in EVENT_NAMES:
                fallback[EVENT_NAMES.index(name)] *= 0.1
        fb_idx = fallback.argmax().item()
        if fallback[fb_idx].item() > 0.01:
            pred_idx = fb_idx
            conf     = probs[pred_idx].item()

    # Update tracker with this prediction
    _prediction_counts[EVENT_NAMES[pred_idx]] += 1

    return pred_idx, conf, probs


print("OK Accuracy boosting helpers defined.")
print(f"   Logit bias    : Corner={cfg.logit_bias[3]:+.1f}  Foul={cfg.logit_bias[1]:+.1f}  "
      f"Shot={cfg.logit_bias[0]:+.1f}  Goal={cfg.logit_bias[2]:+.1f}")
print(f"   Temperature   : {cfg.temperature}")
print(f"   Conf threshold: {cfg.conf_threshold}")
print("   Prediction caps:")
for name, share in _MAX_SHARE.items():
    print(f"     {name:<10}: max {int(share*100)}%")

OK Accuracy boosting helpers defined.
   Logit bias    : Corner=+0.7  Foul=+0.2  Shot=+0.4  Goal=+0.6
   Temperature   : 0.85
   Conf threshold: 0.25
   Prediction caps:
     Shot      : max 25%
     Foul      : max 42%
     Goal      : max 7%
     Corner    : max 10%
     Offside   : max 7%
     Kickoff   : max 9%
     Other     : max 8%


## Cell 10 - Annotation-Guided Evaluator
Extracts one clip per annotated timestamp, runs inference with boosting applied.

In [71]:
@torch.no_grad()
def evaluate_match(match_dir, backbone, event_model, tension_model, cfg, device):
    match_dir = Path(match_dir)
    halves    = find_video_halves(match_dir)
    if len(halves) < 2:
        print(f"  ERROR: Need 2 video files in {match_dir}, found {len(halves)}")
        return []

    anns  = load_annotations(match_dir)
    half1, half2 = halves[0], halves[1]

    total_anns   = sum(len(v) for v in anns.values())
    named_events = sum(1 for v in anns.values() for _, lbl in v if lbl != 6)

    print(f"\nMatch      : {match_dir.name}")
    print(f"Videos     : {half1.name}  |  {half2.name}")
    print(f"Annotations: {total_anns} total  |  {named_events} named events")

    all_results = []

    for video_half, vp in [(1, half1), (2, half2)]:
        half_events = anns.get(video_half, [])
        named = [(sec, lbl) for sec, lbl in half_events if lbl != 6]

        if not named:
            print(f"\n  Half {video_half}: no named annotations - skipping")
            continue

        print(f"\n  Half {video_half} - {vp.name}  ({len(named)} events)")
        print(f"  Time     True Label   Predicted    Conf     Tension  OK?")
        print(f"  " + "-" * 62)

        for event_sec, true_label in sorted(named, key=lambda x: x[0]):
            try:
                frames = extract_event_clip_raw(
                    vp, event_sec,
                    cfg.clip_before_sec, cfg.clip_after_sec,
                    cfg.sample_fps, cfg.img_size)
            except Exception as e:
                log.warning(f"  Clip failed @{event_sec:.0f}s: {e}")
                all_results.append({
                    "match": match_dir.name, "half": video_half,
                    "time": f"{int(event_sec//60):02d}:{int(event_sec%60):02d}",
                    "time_sec": event_sec, "true": EVENT_NAMES[true_label],
                    "predicted": "ERROR", "correct": False,
                    "conf": 0.0, "tension": 0.0, "error": str(e),
                })
                continue

            # Encode frames
            feat = encode_frames(frames, backbone, device, cfg)
            feat = feat.unsqueeze(0).to(device)

            # Optical flow
            flow = (compute_optical_flow(frames).unsqueeze(0).to(device)
                    if cfg.use_optical_flow else None)

            # Raw logits from model
            logits = event_model(feat, flow)  # (1, num_classes)

            # Apply all accuracy boosts
            pred_idx, conf, probs = boosted_predict(
                logits, event_sec, half_events, cfg)

            # Top-3
            top3_idx  = probs.topk(3).indices.tolist()
            top3_conf = probs.topk(3).values.tolist()
            top3_str  = "  ".join(
                f"{EVENT_NAMES[i]}({top3_conf[j]:.0%})"
                for j, i in enumerate(top3_idx))

            # Tension
            tension = tension_model(feat).item()

            mm = int(event_sec // 60)
            ss = int(event_sec % 60)
            ok = "OK" if pred_idx == true_label else "X "

            print(f"  {mm:02d}:{ss:02d}    "
                  f"{EVENT_NAMES[true_label]:<12} "
                  f"{EVENT_NAMES[pred_idx]:<12} "
                  f"{conf:.1%}    "
                  f"{tension:.2f}      {ok}")

            all_results.append({
                "match":    match_dir.name,
                "half":     video_half,
                "time":     f"{mm:02d}:{ss:02d}",
                "time_sec": event_sec,
                "true":     EVENT_NAMES[true_label],
                "predicted":EVENT_NAMES[pred_idx],
                "correct":  pred_idx == true_label,
                "conf":     round(conf, 4),
                "tension":  round(tension, 4),
                "top3":     top3_str,
                "error":    "",
            })

    return all_results


print("OK evaluate_match() defined.")


OK evaluate_match() defined.


## Cell 11 - Run Evaluation

In [72]:
reset_prediction_tracker()   # reset counts before each full evaluation run
all_results = []

for match_dir in TEST_MATCH_DIRS:
    if not Path(match_dir).exists():
        print(f"WARNING: {match_dir} does not exist - skipping.")
        continue
    results = evaluate_match(
        match_dir, backbone, event_model, tension_model, cfg, device)
    all_results.extend(results)

print(f"\n" + "=" * 60)
print(f"  OVERALL RESULTS")
print("=" * 60)

valid  = [r for r in all_results if r["predicted"] != "ERROR"]
errors = [r for r in all_results if r["predicted"] == "ERROR"]

if not valid:
    print("  No results. Check paths and checkpoints.")
else:
    correct = sum(1 for r in valid if r["correct"])
    total   = len(valid)
    print(f"  Events evaluated : {total}")
    print(f"  Correct          : {correct}")
    print(f"  Overall accuracy : {correct/total:.1%}")
    if errors:
        print(f"  Clip errors      : {len(errors)}")
    avg_conf    = sum(r["conf"]    for r in valid) / total
    avg_tension = sum(r["tension"] for r in valid) / total
    print(f"  Avg confidence   : {avg_conf:.1%}")
    print(f"  Avg tension      : {avg_tension:.3f}")
print("=" * 60)


OK Prediction tracker reset.

Match      : match_002
Videos     : 1_720p.mkv  |  2_720p.mkv
Annotations: 176 total  |  84 named events

  Half 1 - 1_720p.mkv  (41 events)
  Time     True Label   Predicted    Conf     Tension  OK?
  --------------------------------------------------------------
  00:00    Kickoff      Kickoff      25.1%    0.21      OK
  02:36    Foul         Goal         44.5%    0.13      X 
  03:41    Shot         Foul         36.0%    0.19      X 
  03:41    Foul         Foul         30.3%    0.19      OK
  05:14    Foul         Goal         29.4%    0.17      X 
  05:28    Foul         Foul         25.9%    0.13      OK
  07:45    Foul         Foul         40.1%    0.19      OK
  08:10    Foul         Foul         31.5%    0.20      OK
  09:23    Shot         Goal         32.5%    0.17      X 
  11:23    Foul         Corner       55.7%    0.12      X 
  11:49    Foul         Corner       18.5%    0.16      X 
  14:33    Shot         Foul         26.8%    0.20      

## Cell 12 - Per-Class Breakdown & Confusion Matrix
**Backslash f-string bug is fixed here.**

In [73]:
valid = [r for r in all_results if r["predicted"] != "ERROR"]

if not valid:
    print("No valid results to analyse.")
else:
    true_c  = Counter(r["true"]      for r in valid)
    corr_c  = Counter(r["true"]      for r in valid if r["correct"])
    pred_c  = Counter(r["predicted"] for r in valid)

    # Per-class accuracy
    print("Per-class accuracy:")
    print(f"  {'Event':<12} {'Correct':>8} {'Total':>7} {'Acc':>7}  Bar")
    print("  " + "-" * 55)
    for name in EVENT_NAMES:
        t = true_c.get(name, 0)
        c = corr_c.get(name, 0)
        if t == 0: continue
        pct = c / t
        bar = ("+" * c) + ("-" * (t - c))
        print(f"  {name:<12} {c:>8} {t:>7} {pct:>7.1%}  [{bar}]")

    # Confusion matrix — FIX: avoid backslash inside f-string
    print("\nConfusion matrix (rows=True, cols=Predicted):")
    active_classes = [n for n in EVENT_NAMES
                      if true_c.get(n, 0) > 0 or pred_c.get(n, 0) > 0]

    confusion = defaultdict(lambda: defaultdict(int))
    for r in valid:
        confusion[r["true"]][r["predicted"]] += 1

    col_w = 8
    # Build header WITHOUT backslash in f-string
    row_label = "True/Pred"
    header = f"  {row_label:<12}" + "".join(f"{n[:col_w]:>{col_w}}" for n in active_classes)
    print(header)
    print("  " + "-" * len(header))
    for true_name in active_classes:
        row = f"  {true_name:<12}"
        for pred_name in active_classes:
            val  = confusion[true_name][pred_name]
            # Bracket diagonal entries (correct predictions)
            cell = f"[{val}]" if (true_name == pred_name and val > 0) else str(val)
            row += f"{cell:>{col_w}}"
        print(row)

    # Most common misclassifications
    mistakes = [(r["true"], r["predicted"])
                for r in valid if not r["correct"]]
    if mistakes:
        print("\nTop misclassifications:")
        for (true_name, pred_name), count in Counter(mistakes).most_common(5):
            print(f"  {true_name:<12} predicted as {pred_name:<12} x{count}")

    # Prediction distribution — shows if Corner bias is fixed
    print("\nPrediction distribution (what the model output):")
    for name in EVENT_NAMES:
        n = pred_c.get(name, 0)
        if n == 0: continue
        bar = chr(9608) * min(n, 40)
        print(f"  {name:<12}: {n:>4}  {bar}")


Per-class accuracy:
  Event         Correct   Total     Acc  Bar
  -------------------------------------------------------
  Shot                2      17   11.8%  [++---------------]
  Foul               26      49   53.1%  [++++++++++++++++++++++++++-----------------------]
  Goal                2       4   50.0%  [++--]
  Corner              1       6   16.7%  [+-----]
  Offside             0       2    0.0%  [--]
  Kickoff             4       6   66.7%  [++++--]

Confusion matrix (rows=True, cols=Predicted):
  True/Pred       Shot    Foul    Goal  Corner Offside Kickoff
  --------------------------------------------------------------
  Shot             [2]       7       3       3       2       0
  Foul               1    [26]      13       8       1       0
  Goal               0       1     [2]       1       0       0
  Corner             0       3       2     [1]       0       0
  Offside            0       1       0       1       0       0
  Kickoff            0       2       0 

## Cell 13 - Generate Commentary

In [74]:
@torch.no_grad()
def generate_commentary(event_idx, tension_val, lm_head, lm, tokenizer, cfg, device):
    state_dim = cfg.num_event_classes + cfg.num_formations + 1
    state = torch.zeros(1, state_dim).to(device)
    state[0, event_idx] = 1.0
    state[0, -1]        = tension_val
    prefix = lm_head(state).unsqueeze(1)
    out = lm.generate(
        inputs_embeds=prefix,
        max_new_tokens=40,
        do_sample=True,
        temperature=0.8,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(out[0], skip_special_tokens=True).strip()


valid = [r for r in all_results if r["predicted"] != "ERROR"]

if lm is None or lm_head is None or tokenizer is None:
    print("Commentary LM not available - skipping.")
else:
    print("Generated commentary:\n")
    print(f"  Time     True         Pred         Tension  Commentary")
    print("  " + "-" * 80)
    for r in valid:
        try:
            pred_idx   = EVENT_NAMES.index(r["predicted"])
            commentary = generate_commentary(
                pred_idx, r["tension"], lm_head, lm, tokenizer, cfg, device)
            r["commentary"] = commentary
        except Exception as e:
            commentary = f"(error: {e})"
            r["commentary"] = commentary
        print(f"  {r['time']:<8} "
              f"{r['true']:<12} "
              f"{r['predicted']:<12} "
              f"{r['tension']:.2f}    "
              f"{commentary}")


Generated commentary:

  Time     True         Pred         Tension  Commentary
  --------------------------------------------------------------------------------
  00:00    Kickoff      Kickoff      0.21    - at the. at the.
  02:36    Foul         Goal         0.13    . attack drift clear the.
  03:41    Shot         Foul         0.19    - outside box def wide a blocked.
  03:41    Foul         Foul         0.19    - got the. at.
  05:14    Foul         Goal         0.17    from the left, centre, and blocked.
  05:28    Foul         Foul         0.13    of the fires at Century Club The fires at - The.
  07:45    Foul         Foul         0.19    - at the. at.
  08:10    Foul         Foul         0.20    of the fires at Daniel in.
  09:23    Shot         Goal         0.17    - at - clear the half.
  11:23    Foul         Corner       0.12    , midfield, attack.
  11:49    Foul         Corner       0.16    corner onetwo a blocked.
  14:33    Shot         Foul         0.20    played saf

## Cell 14 - Export Results to CSV

In [75]:
import time as _time

if not all_results:
    print("No results to export.")
else:
    timestamp = _time.strftime("%Y%m%d_%H%M%S")
    out_path  = Path(cfg.results_dir) / f"evaluation_{timestamp}.csv"

    df = pd.DataFrame(all_results)
    if "commentary" not in df.columns:
        df["commentary"] = ""

    cols = ["match","half","time","time_sec","true","predicted",
            "correct","conf","tension","top3","commentary","error"]
    cols = [c for c in cols if c in df.columns]
    df   = df[cols]
    df.to_csv(out_path, index=False)
    print(f"Results saved to: {out_path}")

    valid_df = df[df["predicted"] != "ERROR"]
    summary  = (
        valid_df.groupby("true")["correct"]
        .agg(total="count", correct="sum")
        .assign(accuracy=lambda x: x["correct"] / x["total"])
        .sort_values("total", ascending=False)
    )
    print("\nPer-class summary:")
    print(summary.to_string())
    print(f"\nPreview:")
    print(df[["match","half","time","true","predicted","correct","conf"]]
          .head(10).to_string(index=False))


Results saved to: /home/sysadm/Desktop/Gamma Force/test_results/evaluation_20260423_162754.csv

Per-class summary:
         total  correct  accuracy
true                             
Foul        49       26  0.530612
Shot        17        2  0.117647
Corner       6        1  0.166667
Kickoff      6        4  0.666667
Goal         4        2  0.500000
Offside      2        0  0.000000

Preview:
    match  half  time    true predicted  correct   conf
match_002     1 00:00 Kickoff   Kickoff     True 0.2511
match_002     1 02:36    Foul      Goal    False 0.4451
match_002     1 03:41    Shot      Foul    False 0.3600
match_002     1 03:41    Foul      Foul     True 0.3030
match_002     1 05:14    Foul      Goal    False 0.2942
match_002     1 05:28    Foul      Foul     True 0.2593
match_002     1 07:45    Foul      Foul     True 0.4010
match_002     1 08:10    Foul      Foul     True 0.3153
match_002     1 09:23    Shot      Goal    False 0.3248
match_002     1 11:23    Foul    Corner    